In [1]:
#Importing Neccessary Libraries
import pandas as pd
import numpy as np

In [3]:
print(f"Data loaded successfully! Shape: {df.shape[0]:,} rows, {df.shape[1]} columns.\n")

Data loaded successfully! Shape: 429,435 rows, 67 columns.



In [4]:
df.columns

Index(['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       't

In [5]:
#Selecting columns of interest for analysis
target_colunmns = [
    'iso_code', 'continent', 'location', 'date', 'population',
    'total_cases', 'total_deaths', 'people_vaccinated', 
    'people_fully_vaccinated', 'total_vaccinations'
]

#Making a copy 
df_subset = df[target_colunmns].copy()

In [6]:
#Removing the continent headers to leave only individual countries
df_clean = df_subset[df_subset['continent'].notna()].copy()

In [7]:
#Converting the date column into a datetime format
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [8]:
#Sorting by country and date to organize the timeline
df_clean = df_clean.sort_values(['location', 'date']).reset_index(drop=True)

In [9]:
# Fill missing data points with the previous day's numbers
metrics = ['total_cases', 'total_deaths', 'people_vaccinated', 'people_fully_vaccinated', 'total_vaccinations']
df_clean[metrics] = df_clean.groupby('location')[metrics].ffill()
df_clean[metrics] = df_clean[metrics].fillna(0)

print(f"Data cleaning finished. Dataset shape: {df_clean.shape} rows and columns respectively")

Data cleaning finished. Dataset shape: (402910, 10) rows and columns respectively


In [10]:
# Checking for any rows where the location and date are exactly identical
duplicate_count = df_clean.duplicated(subset=['location', 'date']).sum()

print(f"Duplicate Check Results:")
print(f"• Found {duplicate_count} duplicate rows in the dataset.")


Duplicate Check Results:
• Found 1408 duplicate rows in the dataset.


In [11]:
# Dropping duplicate country-date rows and keep only the first record
df_clean = df_clean.drop_duplicates(subset=['location', 'date']).reset_index(drop=True)


In [12]:
#Grouping by country and extracting the last row of each country's timeline
df_latest = df_clean.groupby('location').last().reset_index()

print(f"✅ Country snapshot dataset created! Size: {df_latest.shape[0]} unique countries.")

✅ Country snapshot dataset created! Size: 243 unique countries.


In [13]:
# Calculate global totals from the latest snapshots
total_countries = df_latest['location'].nunique()
global_cases = df_latest['total_cases'].sum()
global_deaths = df_latest['total_deaths'].sum()
global_vax = df_latest['people_vaccinated'].sum()

print(f" --- MY GLOBAL COVID-19 METRICS ---")
print(f"• Total Countries Covered: {total_countries}")
print(f"• Global Confirmed Cases:  {global_cases:,.0f}")
print(f"• Global Total Deaths:     {global_deaths:,.0f}")
print(f"• Total People Vaccinated: {global_vax:,.0f}")

 --- MY GLOBAL COVID-19 METRICS ---
• Total Countries Covered: 243
• Global Confirmed Cases:  775,866,783
• Global Total Deaths:     7,057,132
• Total People Vaccinated: 5,685,253,305


In [14]:
#Creating the percentage calculations
df_clean['vaccination_coverage'] = (df_clean['people_vaccinated'] / df_clean['population']) * 100
df_clean['fully_vaccinated_pct'] = (df_clean['people_fully_vaccinated'] / df_clean['population']) * 100
df_clean['fatality_rate'] = (df_clean['total_deaths'] / df_clean['total_cases']) * 100
df_clean = df_clean.fillna(0)

# Step 2: Re-create your latest snapshot dataset so it inherits all these new columns
df_latest = df_clean.groupby('location').last().reset_index()

print("✅ Success! Columns are now fully synced.")
print(f"📋 Verified columns inside df_latest:\n{list(df_latest.columns)}")


✅ Success! Columns are now fully synced.
📋 Verified columns inside df_latest:
['location', 'iso_code', 'continent', 'date', 'population', 'total_cases', 'total_deaths', 'people_vaccinated', 'people_fully_vaccinated', 'total_vaccinations', 'vaccination_coverage', 'fully_vaccinated_pct', 'fatality_rate']


In [15]:
#Top 10 Countries by Vaccination Coverage (%)
top_10_vax = df_latest.nlargest(10, 'vaccination_coverage')[['location', 'vaccination_coverage']]
top_10_vax

,location,vaccination_coverage
81,Gibraltar,129.066316
217,Tokelau,116.376123
175,Qatar,105.827064
227,United Arab Emirates,105.825050
147,Nauru,103.270034
30,Brunei,100.478172
171,Pitcairn,100.000000
124,Macao,97.773670
50,Cuba,96.373343
173,Portugal,95.624094


In [16]:
# 2. Bottom 10 Countries by Vaccination Coverage (%)
bottom_10_vax = df_latest[df_latest['vaccination_coverage'] > 0].nsmallest(10, 'vaccination_coverage')[['location', 'vaccination_coverage']]
bottom_10_vax

,location,vaccination_coverage
33,Burundi,0.286348
240,Yemen,3.116640
167,Papua New Guinea,3.766481
92,Haiti,4.500612
125,Madagascar,9.153015
45,Congo,11.653432
76,Gabon,13.028229
35,Cameroon,13.447231
211,Syria,14.895340
192,Senegal,15.503730


In [17]:
#Top 10 Countries by Total Confirmed Cases
top_10_cases = df_latest.nlargest(10, 'total_cases')[['location', 'total_cases']]
top_10_cases

,location,total_cases
229,United States,103436829.0
42,China,99373219.0
97,India,45041748.0
73,France,38997490.0
79,Germany,38437756.0
28,Brazil,37511921.0
203,South Korea,34571873.0
106,Japan,33803572.0
104,Italy,26781078.0
228,United Kingdom,24974629.0


In [18]:
# 4. Top 10 Countries by Case Fatality Rate (%)
top_10_fatality = df_latest[df_latest['total_cases'] >= 1000].nlargest(10, 'fatality_rate')[['location', 'fatality_rate']]
top_10_fatality

,location,fatality_rate
240,Yemen,18.074508
207,Sudan,7.885237
211,Syria,5.508246
201,Somalia,4.979147
169,Peru,4.881293
61,Egypt,4.811801
136,Mexico,4.390745
26,Bosnia and Herzegovina,4.060783
119,Liberia,3.707440
0,Afghanistan,3.400308


In [19]:
# Print a preview of the Top 10 Vaccination Coverage nations
print("🌍 TOP 10 COUNTRIES BY VACCINATION COVERAGE Preview:")
print(top_10_vax.to_string(index=False))

🌍 TOP 10 COUNTRIES BY VACCINATION COVERAGE Preview:
            location  vaccination_coverage
           Gibraltar            129.066316
             Tokelau            116.376123
               Qatar            105.827064
United Arab Emirates            105.825050
               Nauru            103.270034
              Brunei            100.478172
            Pitcairn            100.000000
               Macao             97.773670
                Cuba             96.373343
            Portugal             95.624094


In [20]:
# Saving the complete clean timeline data to a single CSV file
df_clean.to_csv("covid_dashboard_clean.csv", index=False)

print("🎉 File exported successfully as 'covid_dashboard_clean.csv'!")
print("👉 Open Tableau Public and connect this file to begin Phase 6!")


🎉 File exported successfully as 'covid_dashboard_clean.csv'!
👉 Open Tableau Public and connect this file to begin Phase 6!
